# Merging all the old 2025 task's claims
 

In [1]:
import json

# Input files
input_files = [
    r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\1-train-set-claims.json",
    r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\2-val-set-claims.json",
    r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\3-test-set-claims.json"
]

# Output file
output_file = r"E:\Projects\CheckThat 2026\my data\old_data_merged.json"

merged_claims = []
new_id = 1

for file_path in input_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Assuming each file is a list of claim objects
    for claim in data:
        claim["id"] = new_id
        merged_claims.append(claim)
        new_id += 1

# Save merged file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged_claims, f, ensure_ascii=False, indent=4)

print(f"Total claims merged: {len(merged_claims)}")
print(f"Saved to: {output_file}")

Total claims merged: 16675
Saved to: E:\Projects\CheckThat 2026\my data\old_data_merged.json


# Checking for Duplicates


In [4]:
import json
from collections import defaultdict

file_path = r"E:\Projects\CheckThat 2026\my data\old_data_merged.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

claim_map = defaultdict(list)

for item in data:
    claim_text = item["claim"].strip().lower()
    claim_map[claim_text].append(item["id"])

duplicates = {
    claim: ids
    for claim, ids in claim_map.items()
    if len(ids) > 1
}

print(f"Total claims: {len(data)}")
print(f"Duplicate claim texts found: {len(duplicates)}")

if duplicates:
    print("\nFirst 20 duplicate groups:\n")
    
    for i, (claim, ids) in enumerate(duplicates.items()):
        print(f"Duplicate #{i+1}")
        print(f"IDs: {ids}")
        print(f"Claim: {claim[:200]}")
        print("-" * 80)

        if i >= 19:
            break
else:
    print("No duplicate claims found.")

Total claims: 16675
Duplicate claim texts found: 21

First 20 duplicate groups:

Duplicate #1
IDs: [588, 10965]
Claim: rick scott took $200,000 from a family that leased land for drilling "and now he is trying to hide from it."
--------------------------------------------------------------------------------
Duplicate #2
IDs: [981, 4590]
Claim: says rick scott took $200,000 in campaign contributions from a company that "profited off pollution."
--------------------------------------------------------------------------------
Duplicate #3
IDs: [1590, 1974]
Claim: "there are currently almost one million students who are enrolled in higher education, up from 500,000 in 1994."
--------------------------------------------------------------------------------
Duplicate #4
IDs: [2217, 6047]
Claim: "puerto rico got 91 billion dollars for the hurricane, more money than has ever been gotten for a hurricane before."
--------------------------------------------------------------------------------
Dup

# Merging all the new 2026 task's claims + Resoning_traces

In [1]:
import json

input_files = [
    r"E:\Projects\CheckThat 2026\task 2\data\train.json",
    r"E:\Projects\CheckThat 2026\task 2\data\validation.json",
    r"E:\Projects\CheckThat 2026\task 2\data\test.json"
]

output_file = r"E:\Projects\CheckThat 2026\my data\new_data_merged.json"

merged_data = []
new_id = 1

for file_path in input_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data:
        merged_data.append({
            "id": new_id,
            "claim": item["claim"],
            "Reasoning_traces": item["Reasoning_traces"]
        })
        new_id += 1

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=4)

print(f"Saved {len(merged_data)} records to:")
print(output_file)

Saved 10558 records to:
E:\Projects\CheckThat 2026\my data\new_data_merged.json


# Fuzzy Similarity Check

In [ ]:
pip install sentence-transformers faiss-cpu tqdm numpy

In [3]:
import json
import numpy as np
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# --------------------------------------------------
# Files
# --------------------------------------------------

OLD_FILE = r"E:\Projects\CheckThat 2026\my data\old_data_merged.json"
NEW_FILE = r"E:\Projects\CheckThat 2026\my data\new_data_merged.json"

OUTPUT_FILE = r"E:\Projects\CheckThat 2026\my data\semantic_matches.json"

SIMILARITY_THRESHOLD = 0.95

# --------------------------------------------------
# Load data
# --------------------------------------------------

with open(OLD_FILE, "r", encoding="utf-8") as f:
    old_data = json.load(f)

with open(NEW_FILE, "r", encoding="utf-8") as f:
    new_data = json.load(f)

# --------------------------------------------------
# Extract claims
# --------------------------------------------------

old_claims = [x["claim"] for x in old_data]
new_claims = [x["claim"] for x in new_data]

print(f"Old claims: {len(old_claims)}")
print(f"New claims: {len(new_claims)}")

# --------------------------------------------------
# Load model
# --------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")

# --------------------------------------------------
# Encode claims
# --------------------------------------------------

print("Encoding old claims...")

old_embeddings = model.encode(
    old_claims,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Encoding new claims...")

new_embeddings = model.encode(
    new_claims,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# --------------------------------------------------
# Build FAISS index
# --------------------------------------------------

dimension = old_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(old_embeddings)

print("FAISS index built")

# --------------------------------------------------
# Search
# --------------------------------------------------

k = 1

scores, indices = index.search(new_embeddings, k)

matches = []

for i in tqdm(range(len(new_claims)), desc="Finding matches"):

    similarity = float(scores[i][0])

    if similarity >= SIMILARITY_THRESHOLD:

        old_idx = int(indices[i][0])

        matches.append({
            "new_id": new_data[i].get("id"),
            "old_id": old_data[old_idx].get("id"),
            "similarity": round(similarity, 4),
            "new_claim": new_claims[i],
            "old_claim": old_claims[old_idx]
        })

print(f"\nMatches >= {SIMILARITY_THRESHOLD}: {len(matches)}")

# --------------------------------------------------
# Save
# --------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(matches, f, indent=4, ensure_ascii=False)

print(f"Saved to:\n{OUTPUT_FILE}")

Old claims: 16675
New claims: 10558


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding new claims...


Batches:   0%|          | 0/83 [00:00<?, ?it/s]

FAISS index built


Finding matches: 100%|██████████| 10558/10558 [00:00<00:00, 455835.03it/s]


Matches >= 0.95: 6469
Saved to:
E:\Projects\CheckThat 2026\my data\semantic_matches.json


# Data Distriution

In [11]:
import json
import pandas as pd

file_path = r"E:\Projects\CheckThat 2026\my data\semantic_matches.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

sims = [x["similarity"] for x in data]

bins = [
    0.95, 0.96,
    0.97,
    0.98,
    0.99,
    0.9999999,
    1.001
]

labels = [
    "95-96%",
    "96-97%",
    "97-98%",
    "98-99%",
    "99-99.99999%",
    "100-100%"
]

bucket_counts = pd.cut(
    sims,
    bins=bins,
    labels=labels,
    include_lowest=True
).value_counts().sort_index()

print(bucket_counts)

95-96%             5
96-97%             4
97-98%            13
98-99%            19
99-99.99999%      26
100-100%        6402
Name: count, dtype: int64


# Summary:
----------------------------------------
95-99.99% Match -- 67 

100% Match ------- 6402

----------------------------------------

